# Qwen3-0.6B × 커스텀 Triton AttentionBackend (varlen, vLLM v1 실제 방식)

`vllm_unified` 가 "수학은 v1 스타일이지만 grid launch 는 seq-first" 였다면,
이 프로젝트는 **grid 까지 varlen 으로** — vLLM v1 의 `kernel_unified_attention_2d` 와 동일한 패턴.

**이 프로젝트의 특징**:
- 커널 수학은 unified 그대로 (절대 위치 causal mask 로 prefill/decode/chunked 통합)
- **Grid 가 `(total_q_blocks_rounded, Hq)`** — 시퀀스 축이 grid 에서 사라짐
- **`find_seq_idx` binary search** 로 각 program 이 자기 담당 시퀀스 탐색
- **Seq-aligned flat**: 각 seq 의 q 토큰이 BLOCK_Q 경계로 round-up 되어 flat 하게 펼쳐짐. 한 BLOCK_Q 안에는 항상 한 seq 만.


## 준비물 & 설치

- CUDA GPU 필수
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)

```bash
# 노트북 디렉토리에서
pip install -e .
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = triton_attention_backend:register`) 가
모든 vLLM 프로세스(메인·엔진 코어·워커) 에서 자동으로 `register()` 를 호출해 준다.
즉 **노트북 안에서 `register()` 를 수동 호출하지 않는다**.

절차:
1. cell 3 — 환경 확인
2. cell 8 — 커널 smoke test
3. cell 10 — LLM 생성 (plugin 이 이미 CUSTOM 슬롯 점유)
4. cell 11~13 — generate + 실행 증거 확인


In [ ]:
import torch, triton, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('triton:', triton.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

# 이 노트북은 vLLM 0.19.x 기준으로 작성됨. 다른 버전에서는 내부 API drift 가능.
assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)


## Qwen3-0.6B 구조 요약

| 항목 | 값 |
|---|---|
| hidden_size | 1024 |
| Q heads | 16 |
| KV heads | 8 (GQA 2:1) |
| head_dim | 128 |
| layers | 28 |

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

## Varlen 커널의 grid 토폴로지

`triton_attn.py` 의 `_fwd_kernel_varlen`:

```
Grid: (total_q_blocks_rounded, num_query_heads)

q_block_global_idx = program_id(0)
q_head_idx         = program_id(1)

seq_idx = find_seq_idx(cum_q_blocks, q_block_global_idx, num_seqs)  # binary search
q_block_local_idx = q_block_global_idx - cum_q_blocks[seq_idx]

# 한 프로그램: 이 seq 의 q_block_local_idx 번째 BLOCK_Q 토큰 × 1 Q head
```

**`cum_q_blocks`** 는 각 seq 의 `cdiv(q_len, BLOCK_Q)` 의 누적합:
```
예: q_lens = [5, 1, 1],  BLOCK_Q = 4
    blocks_per_seq      = [2, 1, 1]
    cum_q_blocks        = [0, 2, 3, 4]
    total_q_blocks      = 4

program 0 (block=0) → seq 0, tokens 0..3  (BLOCK_Q 로 full)
program 1 (block=1) → seq 0, tokens 4..4  (나머지 1 토큰 + padding 3)
program 2 (block=2) → seq 1, token 0     (BLOCK_Q 중 1 유효, 3 padding)
program 3 (block=3) → seq 2, token 0
```

**seq-first 와 비교 — program 수**:
- seq-first: `num_seqs × Hq × max_q_blocks` = 3 × Hq × 2 = 6×Hq  
  (가장 긴 seq 기준으로 잡혀 seq1/seq2 가 idle program 생성)
- varlen: `total_q_blocks × Hq` = 4×Hq  
  (각 seq 마다 실제 필요한 만큼만)


In [ ]:
from triton_attn import triton_attention_varlen, _pack_to_paged_multiseq, _find_seq_idx
print(triton_attention_varlen.__doc__)


In [ ]:
# Smoke test: varlen grid + ultimate mix (prefill + chunked + decode 한 배치)
import torch
import torch.nn.functional as F

Hq, Hkv, D = 16, 8, 128
dtype = torch.float16
specs = [(64, 64), (16, 128), (1, 200), (1, 50)]   # prefill / chunked / decode / decode

full_q_per_seq, k_per_seq, v_per_seq, ref_per_seq = [], [], [], []
for q_len, s_len in specs:
    q_full = torch.randn(Hq, s_len, D, dtype=dtype, device='cuda')
    k_s = torch.randn(Hkv, s_len, D, dtype=dtype, device='cuda')
    v_s = torch.randn(Hkv, s_len, D, dtype=dtype, device='cuda')
    full_q_per_seq.append(q_full); k_per_seq.append(k_s); v_per_seq.append(v_s)
    k_rep = k_s.repeat_interleave(Hq // Hkv, dim=0)
    v_rep = v_s.repeat_interleave(Hq // Hkv, dim=0)
    ref = F.scaled_dot_product_attention(
        q_full.unsqueeze(0), k_rep.unsqueeze(0), v_rep.unsqueeze(0), is_causal=True
    ).squeeze(0)
    ref_per_seq.append(ref[:, s_len - q_len:, :])

chunks = [full_q_per_seq[i][:, -specs[i][0]:, :].transpose(0, 1) for i in range(len(specs))]
q_flat = torch.cat(chunks, dim=0)
q_lens = [s[0] for s in specs]; s_lens = [s[1] for s in specs]
cumulative = torch.cumsum(torch.tensor(q_lens, dtype=torch.int32), 0)
query_start_loc = torch.cat([torch.tensor([0], dtype=torch.int32), cumulative]).to('cuda')
seq_lens_t = torch.tensor(s_lens, dtype=torch.int32, device='cuda')

key_cache, value_cache, block_table = _pack_to_paged_multiseq(k_per_seq, v_per_seq, block_size=16)
ours = triton_attention_varlen(q_flat, key_cache, value_cache, block_table, seq_lens_t, query_start_loc)

max_err = 0.0
for i, q_len in enumerate(q_lens):
    start = int(query_start_loc[i].item())
    ours_seq = ours[start:start + q_len].transpose(0, 1)
    err = (ours_seq.float() - ref_per_seq[i].float()).abs().max().item()
    max_err = max(max_err, err)

print(f'varlen kernel prefill+chunked+decode specs={specs}')
print(f'  max_abs_err = {max_err:.6f}  →  {"PASS" if max_err < 1e-2 else "FAIL"}')


## Plugin + varlen launch 로직

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_varlen_backend = "triton_attention_backend:register"
```

**Backend 는 unified 와 완전히 동일한 한 줄 호출**:
```python
o_flat = triton_attention_varlen(
    query[:N],
    key_cache, value_cache,
    block_table, seq_lens, query_start_loc,
    scale=self.scale,
)
```

차이는 **래퍼 내부에서** 일어남 — `cum_q_blocks = cumsum(cdiv(q_lens, BLOCK_Q))` 계산 후
`grid = (total_q_blocks, Hq)` 로 launch. 커널 안에서 `find_seq_idx` binary search 로
각 program 이 자기 seq 를 찾음.


In [ ]:
# entry point 등록 상태 확인
from importlib.metadata import entry_points
eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_varlen_backend']
assert mine, '`my_varlen_backend` entry point 가 보이지 않는다. pip install -e . 확인.'
print('plugin registered:', mine[0].name, '->', mine[0].value)


In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

# max_num_batched_tokens=64 로 낮춰서 chunked prefill 을 강제로 trigger.
# 이보다 긴 prompt 가 들어오면 vLLM 이 prefill 을 64-token chunk 로 쪼갬.
# 기본값 (8192) 에서는 웬만한 prompt 가 한 번에 들어가서 chunking 관찰 불가.
llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=4,
    max_model_len=2048,
    max_num_batched_tokens=64,   # ← chunked prefill trigger 포인트
)


In [ ]:
# 짧은 prompt 3개 + 긴 prompt 1개 (100+ 토큰).
# 긴 prompt 는 max_num_batched_tokens=64 보다 길어서 여러 chunk 로 쪼개짐 → chunked prefill 발동.
long_prompt = 'In the long history of artificial intelligence research, from the early symbolic AI of the 1950s through the neural network revival of the 1980s, the deep learning breakthroughs of the 2010s, and the transformer-based large language models of the 2020s, one theme has remained constant: ' + 'the answer is'

prompts = [
    'The capital of France is',
    long_prompt,
    'Shakespeare wrote the play',
    'Python was created by',
]
out = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=16))
for i, o in enumerate(out):
    print(f'[{i}] (prompt {len(prompts[i])} chars) {o.outputs[0].text[:80]}')


## 검증 — 이제 `chunked > 0` 이 보여야 한다

엔진 코어 stderr (cell 11 출력 바로 위) 에 다음 같은 로그들이 찍혔어야:

```
fired (...) num_seqs=... prefill-like=... decode=... chunked=C max_q_len=... tokens=...
```

**이 실험에서 중요한 건 `chunked=C > 0`**. `max_num_batched_tokens=64` + 긴 prompt 조합이
vLLM 스케줄러로 하여금 긴 prompt 의 prefill 을 여러 chunk 로 쪼개게 만들고, 그 중간 chunk
들은 `1 < q_len < s_len` 상태 → backend 에 `chunked` 로 집계됨.

**관찰 포인트**:
- 긴 prompt 의 처음 chunk: `q_len=64, s_len=64` → prefill-like (pure prefill)
- 긴 prompt 의 두 번째 chunk: `q_len=64, s_len=128` → **chunked** (1 < q_len < s_len)
- 동시에 짧은 prompt 들의 decode 가 섞일 수도 있음 → 한 forward 에 prefill+chunked+decode 공존

**chunked prefill 이 유의미한 이유**:
긴 prompt 가 다른 시퀀스의 decode 를 막지 않음. 64 토큰씩 쪼개 처리되는 동안 다른 시퀀스의
decode 가 같은 forward 에 함께 돌아가 → latency 균등화 (head-of-line blocking 회피).


In [ ]:
# 노트북 프로세스에서 확인 가능한 부분
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyTritonBackend' in path
print('OK — plugin 자동 등록 확인')
print()
print('unified 커널 실행 증거 = 위 cell 11 출력 바로 위의 `fired (unified)` 로그')


## 완주

```
padded_decode → split → paged → multiseq → unified → varlen (← 여기, vLLM v1 실제 방식)
```

vLLM v1 의 `kernel_unified_attention_2d` 와 이제 동일 구조:
- Flat-packed Q + varlen meta (cu_seqlens 스타일)
- Seq-aligned flat grid + `find_seq_idx` binary search
- 통합 수학 (절대 위치 causal)

남은 advanced 최적화 (선택 학습):
- **Split-k (stage1 + stage2)** — 긴 컨텍스트 decode 의 softmax reduction 분할
- **GQA 를 grid 로 접기** — `(total_q_blocks, num_kv_heads)` + 프로그램 내부 n_rep 처리 (vLLM v1 의 실제 축)
- **fp8 KV cache**, **cudagraph**, **sliding_window / alibi / sinks** 등 기능 확장
